# Importing Libraries

In [ ]:
import hickle
import pandas as pd
import scipy.signal

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from scipy import ndimage
from tqdm import tqdm
import corner
from PIL import Image
import io

import os

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from scipy.stats import chi2
from sklearn.datasets import make_s_curve
import matplotlib as mpl
import matplotlib.pyplot as plt
import tqdm

import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PosteriorCoverage, PlotSinglePosterior

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from datasets import load_dataset, DatasetDict
from transformers import ViTFeatureExtractor, ViTForImageClassification, ViTModel, AdamW, AutoImageProcessor, AutoModel
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from tqdm import tqdm


# Data Importing

In [ ]:
def check_degeneracy(samples, vb=False):
    # Ensure only the 6 physical parameters are used if 34D samples are passed
    samples_6d = samples[:, 0:6] if samples.shape[1] > 6 else samples
    covs = np.zeros((samples_6d.shape[1], samples_6d.shape[1]))
    pvals = np.ones((samples_6d.shape[1], samples_6d.shape[1]))
    for i in range(samples_6d.shape[1]):
        for j in range(i):
            covs[i,j], pvals[i,j] = spearmanr(samples_6d[:,i], samples_6d[:,j])

    if vb==True:
        plt.imshow(np.abs(covs[1:,:-1]));
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()
        
        plt.imshow(np.abs((pvals<0.05)[1:,:-1]))
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()

    return covs[1:,:-1], pvals[1:,:-1]

In [ ]:
GSWLC_data = load_dataset('juanpabloalfonzo/GSWLC_Data',split='train')
gswlc_data = GSWLC_data.to_pandas()

gswlc_images_skyserver = load_dataset('juanpabloalfonzo/GSWLC_SkyServer',split='train')
gswlc_images_skyserver = gswlc_images_skyserver['images'].copy()

timescales_data_cami_fixed = np.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/all_timescales_cami_fixed_v2.npy', allow_pickle=True).item()


train_index = int(0.8*len(gswlc_data))
sample_size = len(gswlc_data) - train_index

In [ ]:
morph_data = load_dataset('juanpabloalfonzo/GSWLC_morph',split='train')
morph_data = morph_data.to_pandas()

if 'ID' not in gswlc_data.columns:
        print("\nError: 'ID' column not found in gswlc_data.")
elif 'ID' not in morph_data.columns:
    print("\nError: 'ID' column not found in morph_data.")

elif 'bt_morph' not in morph_data.columns:
        print("\nError: 'bt_morph' column not found in morph_data.")
else:
    print("\nData loaded successfully. Proceeding with merge...")
    gswlc_data_merged = pd.merge(gswlc_data, morph_data[['ID', 'pbulge_morph','bt_morph','half_light_radius_morph', 'bulge_n_morph', 'disk_n_morph', 'ttype_morph', 'pbar_morph', 'pdisk_morph', 'pedgeon_morph', 'pmerger_morph']], on='ID', how='left')

sersic_n = np.array(gswlc_data_merged['bulge_n_morph'][train_index:])
bt = np.array(gswlc_data_merged['bt_morph'][train_index:])
pbulge = np.array(gswlc_data_merged['pbulge_morph'][train_index:])
ttype= np.array(gswlc_data_merged['ttype_morph'][train_index:])
pbar = np.array(gswlc_data_merged['pbar_morph'][train_index:])
pdisk = np.array(gswlc_data_merged['pdisk_morph'][train_index:])
pedgeon = np.array(gswlc_data_merged['pedgeon_morph'][train_index:])
pmerger = np.array(gswlc_data_merged['pmerger_morph'][train_index:])


In [ ]:
temp_fuv = np.array(gswlc_data['FUV']).copy()
temp_fuv[temp_fuv > 25] = 25 + 0.3*np.random.normal(size=(np.sum(temp_fuv>25),))

temp_w2 = np.array(gswlc_data['W2']).copy()
temp_w2[temp_w2 > 23] = 23

temp_w3 = np.array(gswlc_data['W3']).copy()
temp_w3[temp_w3 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w3>23),))

temp_w4 = np.array(gswlc_data['W4']).copy()
temp_w4[temp_w4 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w4>23),))

eleven_bands_SED = np.array((temp_fuv, 
               gswlc_data['NUV'],
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               gswlc_data['W1'],
               temp_w2,
               temp_w3,
               temp_w4,)).T

tempt50 = np.array(gswlc_data['t_50']).copy()
tempt50[(tempt50 > 13.7)] = 13.7

tempt10 = np.array(gswlc_data['t_90']).copy()
tempt10[(tempt10 > 13.7)] = 13.7

tempt90 = np.array(gswlc_data['t_90']).copy()
tempt90[(tempt90 > 13.7)] = 13.7

tempssfr = np.array(gswlc_data['logSSFR_50']).copy()
tempssfr[tempssfr < -3.5] = tempssfr[tempssfr < -3.5] + 0.3*np.random.normal(size=(np.sum(tempssfr < -3.5),))

tempmet = np.array(gswlc_data['logOH_50']).copy()
tempmet[tempmet < 7.1] = tempmet[tempmet < 7.1] + 0.1*np.random.normal(size=(np.sum(tempmet < 7.1),))

five_bands_SED= np.array((
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               )).T

def safe_log_sfr(arr):
            return np.log10(np.clip(arr, 1e-3, None))

np.random.seed(256)
theta_full_sfh_sfr = np.vstack((
    gswlc_data['logM_50_x'], 
    tempssfr,
    gswlc_data['specz'],
    tempt50,
    gswlc_data['tauV_50'],
    tempmet,
    tempt10,
    tempt90,
    safe_log_sfr(timescales_data_cami_fixed['now']['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.1]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.3]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.7]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.8]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.3]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.7]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[2.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[2.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[3.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[3.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[4.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[4.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[5.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[5.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[6.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[6.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[7.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[7.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[8.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[8.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[9.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[9.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[10.0]['sfr'])
)).T

theta_full_sfh_sfr[np.isnan(theta_full_sfh_sfr)] = 0.
theta_full_sfh_sfr[:,8:][theta_full_sfh_sfr[:,8:]<-2.5] = theta_full_sfh_sfr[:,8:][theta_full_sfh_sfr[:,8:]<-2.5] + 0.3*np.random.normal(size=(np.sum(theta_full_sfh_sfr[:,8:]<-2.5),))

theta=theta_full_sfh_sfr[:, 0:6]

In [ ]:
#Importing trained ILI models 
ili_dir = "INSERT PATH TO ILI MODELS HERE"

#Image Models
posterior_ensemble_finetuned_vit, summaries_finetuned_vit = hickle.load(f'{ili_dir}/posterior_ensemble_image_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_half_size_im, summaries_half_size_im = hickle.load(f'{ili_dir}/posterior_ensemble_half_size_image_full_train.hickle')
posterior_ensemble_quarter_size_im, summaries_quarter_size_im = hickle.load(f'{ili_dir}/posterior_ensemble_quarter_size_image_full_train.hickle')
posterior_ensemble_g_band_im, summaries_g_band_im = hickle.load(f'{ili_dir}/posterior_ensemble_g_band_image_full_train.hickle')

#Photometry Models
posterior_ensemble_sed_5_bands , summaries_sed_5_bands = hickle.load(f'{ili_dir}/posterior_ensemble_5_band_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_sed_11_bands , summaries_sed_11_bands = hickle.load(f'{ili_dir}/posterior_ensemble_11_band_full_sfh_sfr_v2_full_train.hickle')


#Combined Models
posterior_ensemble_combined_11_bands, summaries_combined_11_bands = hickle.load(f'{ili_dir}/posterior_ensemble_combined_11_band_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_combined_5_bands_finetuned_vit, summaries_combined_5_bands_finetuned_vit = hickle.load(f'{ili_dir}/posterior_ensemble_combined_5_band_full_sfh_sfr_v2_full_train.hickle')


#Importing ViT embeddings
embeddings_dir = "INSERT PATH TO ViT EMBEDDINGS HERE"
gswlc_embeds_half_size = hickle.load(f'{embeddings_dir}/gswlc_emebds_half_size_clean.hickle')
gswlc_embeds_quarter_size = hickle.load(f'{embeddings_dir}/gswlc_emebds_quarter_size_clean.hickle')
gswlc_embeds_g_band = hickle.load(f'{embeddings_dir}/gswlc_emebds_g_band_clean.hickle')
gswlc_finetuned_embeds = hickle.load(f'{embeddings_dir}/finetuned_embeds.hickle')


# Extracting Predictions & Covariances (Demo Loop with Image Model)

In [ ]:
# all_covs_finetuned_vit_full, all_samples_finetuned_vit_full = [], []
# for gid in tqdm(range(len(gswlc_data))):
#     xobs_im = gswlc_finetuned_embeds[gid,:]
#     x_truth = theta[gid,:]

#     posterior_samples_finetuned_vit = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars = False)
#     posterior_samples_finetuned_vit = posterior_samples_finetuned_vit.cpu().numpy()
#     covs_finetuned_vit, _ = check_degeneracy(posterior_samples_finetuned_vit)

#     all_covs_finetuned_vit_full.append(covs_finetuned_vit)
#     all_samples_finetuned_vit_full.append(posterior_samples_finetuned_vit)

# all_covs_finetuned_vit_full = np.array(all_covs_finetuned_vit_full)
# all_samples_finetuned_vit_full = np.array(all_samples_finetuned_vit_full)

# hickle.dump((all_covs_finetuned_vit_full, all_samples_finetuned_vit_full), '/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/all_covs_and_samples_finetuned_vit_full.hickle')


In [ ]:
SUMMARY_DIR = "INSERT PATH TO POSTERIOR SUMMARIES HERE" # used in paper

def load_summary(filename):
    path = os.path.join(SUMMARY_DIR, filename)
    if os.path.exists(path):
        data = np.load(path)
        return data['p16'], data['p50'], data['p84'], data['covs']
    else:
        print(f"File not found: {path}")
        return None, None, None, None

# ==============================================================================
# 1. LOAD LIGHTWEIGHT SUMMARIES (p16, p50, p84, covs)
# ==============================================================================
# Primary & SFH Models
im_p16, im_p50, im_p84, covs_image = load_summary('summary_image.npz')
sed5_p16, sed5_p50, sed5_p84, covs_5_band = load_summary('summary_5_band.npz')
comb5_p16, comb5_p50, comb5_p84, covs_combined_5_band = load_summary('summary_combined_5_band.npz')

# 11-Band Models
sed11_p16, sed11_p50, sed11_p84, covs_11_band = load_summary('summary_11_band.npz')
comb11_p16, comb11_p50, comb11_p84, covs_combined_11_band = load_summary('summary_combined_11_band.npz')

# Image Ablation Models
gband_p16, gband_p50, gband_p84, covs_g_band = load_summary('summary_g_band.npz')
half_p16, half_p50, half_p84, covs_half_size_im = load_summary('summary_half_size.npz')
quart_p16, quart_p50, quart_p84, covs_quarter_size_im = load_summary('summary_quarter_size.npz')


# # ==============================================================================
# # 2. PRE-COMPUTE VARIABLES NEEDED FOR PLOTTING 
# # (Replaces the RAM-heavy np.percentile calls in the original notebook)
# # ==============================================================================
# # Ensure truth array matches evaluation length (e.g. slicing train_index if necessary)
n_eval_5b = len(sed5_p50)
truth_eval_5b = theta 

# # A. Prediction Errors (Absolute Residuals for the 6 Physical Parameters)
allimerr_full_new = np.abs(im_p50[-n_eval_5b:, 0:6] - truth_eval_5b)
allsederr_5_bands_full_new = np.abs(sed5_p50[:, 0:6] - truth_eval_5b)
allsederr_combined_5_bands_finetuned_vit_full_new = np.abs(comb5_p50[:, 0:6] - truth_eval_5b)

# B. 1-Sigma Credible Intervals ( (P84 - P16) / 2 )
primary_individual_scatters = {
    "Finetuned ViT": (im_p84[-n_eval_5b:, 0:6] - im_p16[-n_eval_5b:, 0:6]) / 2.0,
    "5 Band Photometry": (sed5_p84[:, 0:6] - sed5_p16[:, 0:6]) / 2.0,
    "Combined (ViT + Bands)": (comb5_p84[:, 0:6] - comb5_p16[:, 0:6]) / 2.0
}

primary_individual_scatters_11_bands = {
    "11 Band Photometry": (sed11_p84[:, 0:6] - sed11_p16[:, 0:6]) / 2.0,
    "11 Band Combined": (comb11_p84[:, 0:6] - comb11_p16[:, 0:6]) / 2.0
}

# C. Median Covariance Matrices (for Degeneracy Corner Plots)
median_cov_images = np.nanmedian(covs_image, axis=0)
median_cov_bands = np.nanmedian(covs_5_band, axis=0)
median_cov_combined = np.nanmedian(covs_combined_5_band, axis=0)

mvals_full = theta[:,0]
sfrvals_full = tempssfr + gswlc_data['logM_50_x'] - 9

print("Successfully loaded summaries and computed plotting variables.")

# Tables 1 & 2: Population Bias ($\mu$) and Scatter ($\sigma_{\mathrm{NMAD}}$)

In [ ]:
param_names = ['log M*', 'log SFR', 'z', 't50', 'Av', '12+log(O/H)']

In [ ]:
def calculate_population_metrics(p16, p50, p84, truth):
    """
    Calculates Bias (median residual), NMAD Scatter, and 68% Coverage for the 6 physical parameters.
    Inputs should be the precomputed percentiles of shape (N_gal, 6).
    """
    residuals = p50 - truth
    bias = np.nanmedian(residuals, axis=0)
    
    # NMAD = 1.4826 * median(|residuals - median(residuals)|)
    nmad = 1.4826 * np.nanmedian(np.abs(residuals - bias), axis=0)
    
    is_covered = (truth >= p16) & (truth <= p84)
    coverage_pct = np.nanmean(is_covered, axis=0) * 100
    
    return bias, nmad, coverage_pct

# --- TABLE 1: Imaging vs 5-Band Photometry vs Combined ---
print("=== TABLE 1: Primary Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8} | {'Coverage'}")
print("-" * 75)

# Map directly to the lightweight precomputed summaries (sliced to the 6 physical parameters)
table1_models = {
    "Imaging": (im_p16[:, 0:6], im_p50[:, 0:6], im_p84[:, 0:6]),
    "5-band Photometry": (sed5_p16[:, 0:6], sed5_p50[:, 0:6], sed5_p84[:, 0:6]),
    "Combined": (comb5_p16[:, 0:6], comb5_p50[:, 0:6], comb5_p84[:, 0:6])
}

for model_name, (p16_arr, p50_arr, p84_arr) in table1_models.items():
    truth_to_use = theta 
    bias_arr, scatter_arr, cov_arr = calculate_population_metrics(p16_arr, p50_arr, p84_arr, truth_to_use)
    
    for i, param in enumerate(param_names):
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f} | {cov_arr[i]:>5.1f}%")
    print("-" * 75)

In [ ]:
# --- 11-Band Photometry & Combined Metrics (Appendix C / D) ---
print("=== 11-Band Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8} | {'Coverage'}")
print("-" * 75)

# Map directly to the lightweight precomputed summaries (sliced to the 6 physical parameters)
table_11b_models = {
    "Finetuned ViT": (im_p16[:, 0:6], im_p50[:, 0:6], im_p84[:, 0:6]),
    "11 Band Photometry": (sed11_p16[:, 0:6], sed11_p50[:, 0:6], sed11_p84[:, 0:6]),
    "11 Band Combined": (comb11_p16[:, 0:6], comb11_p50[:, 0:6], comb11_p84[:, 0:6])
}

all_metrics_11_bands = {}
for model_name, (p16_arr, p50_arr, p84_arr) in table_11b_models.items():
    all_metrics_11_bands[model_name] = {}
    truth_to_use = theta if len(p50_arr) == len(theta) else theta[train_index:]
    
    
    # Calculate metrics using the new function signature
    bias_arr, scatter_arr, cov_arr = calculate_population_metrics(p16_arr, p50_arr, p84_arr, truth_to_use)
    
    for i, param in enumerate(param_names):
        all_metrics_11_bands[model_name][param] = {'bias': bias_arr[i], 'scatter': scatter_arr[i], 'coverage': cov_arr[i]}
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f} | {cov_arr[i]:>5.1f}%")
    print("-" * 75)

In [ ]:
# --- TABLE 2: Image Ablation Models (gri vs g-band vs Half-Size vs Quarter-Size) ---
print("=== TABLE 2: Image Ablation Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8}")
print("-" * 60)

# Map directly to the lightweight precomputed summaries (sliced to the 6 physical parameters)
table2_models = {
    "gri Image": (im_p16[:, 0:6], im_p50[:, 0:6], im_p84[:, 0:6]),
    "g-Band Image": (gband_p16[:, 0:6], gband_p50[:, 0:6], gband_p84[:, 0:6]),
    "Half-Size Image": (half_p16[:, 0:6], half_p50[:, 0:6], half_p84[:, 0:6]),
    "Quarter-Size Image": (quart_p16[:, 0:6], quart_p50[:, 0:6], quart_p84[:, 0:6])
}

ablation_metrics = {}
for model_name, (p16_arr, p50_arr, p84_arr) in table2_models.items():
    ablation_metrics[model_name] = {}
    truth_to_use = theta if len(p50_arr) == len(theta) else theta[train_index:]
    
    # Calculate metrics using the new function signature from Table 1
    bias_arr, scatter_arr, _ = calculate_population_metrics(p16_arr, p50_arr, p84_arr, truth_to_use)
    
    for i, param in enumerate(param_names):
        ablation_metrics[model_name][param] = {'bias': bias_arr[i], 'scatter': scatter_arr[i]}
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f}")
    print("-" * 60)

# Figure 1: Distribution of Dataset in $M_*$-SFR Space

In [ ]:
np.random.seed(42)
random_gals_sample = np.random.choice(len(gswlc_data), 100, replace=False)
plt.figure(figsize=(10, 8))
plt.hist2d(gswlc_data['logM_50_x'], gswlc_data['logSFR_50'], bins=30, cmap='Blues', vmax=2500)
plt.errorbar(gswlc_data['logM_50_x'][random_gals_sample], gswlc_data['logSFR_50'][random_gals_sample], 
             yerr=(gswlc_data['logSFR_84'][random_gals_sample]-gswlc_data['logSFR_50'][random_gals_sample],
                   gswlc_data['logSFR_50'][random_gals_sample]-gswlc_data['logSFR_16'][random_gals_sample]), 
             xerr=(gswlc_data['logM_84'][random_gals_sample]-gswlc_data['logM_50_x'][random_gals_sample],
                   gswlc_data['logM_50_x'][random_gals_sample]-gswlc_data['logM_16'][random_gals_sample]), 
             marker='.', markersize=3, alpha=0.4, lw=0, elinewidth=2, capsize=3, color='black', label='100 Random Galaxies')
plt.ylim(-3.1, 3)
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
plt.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')
plt.colorbar(label='Number Density')
plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)
plt.ylabel('log SFR [$M_{\odot}$/yr]', fontsize=18)
plt.title('Distribution of Dataset in $M_*$-SFR Space', fontsize=20)
plt.legend(fontsize=14)
plt.show()

# Figure 3: Inferred Posteriors (5-Band Photometry vs. Image vs. Combined)

In [ ]:
import matplotlib.patches as mpatches

clrs = plt.cm.Spectral(np.linspace(0, 1, 10))
ranges = [(9, 12), (-14, -7), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

np.random.seed(256)
random_gal = np.random.randint(train_index, len(gswlc_data), size=(1))
# random_gal = np.random.randint(train_index, len(small_dif_mask), size=(1))
gid = int(random_gal[0])

xobs_sed = five_bands_SED[gid, :]
xobs_im = gswlc_finetuned_embeds[gid, :]
x_truth_raw = theta[gid, :].copy()
x_truth = x_truth_raw

x_truth[1] = x_truth_raw[1] - 9  # Adjust log SFR for plotting

# Sample from models and slice [:, 0:6] for the 6 physical parameters
posterior_samples_sed = posterior_ensemble_sed_5_bands.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_im = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_combined = posterior_ensemble_combined_5_bands_finetuned_vit.sample((1000,), torch.Tensor(np.hstack((xobs_im, xobs_sed))).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]


plt.imshow(Image.open(io.BytesIO(gswlc_images_skyserver[gid])))
plt.axis('off')
plt.figure()
plt.scatter(np.arange(5), xobs_sed)
plt.xticks(np.arange(0, 5), ['u', 'g', 'r', 'i', 'z'], fontsize=18)
plt.xlabel('Band', fontsize=18)
plt.ylabel('Magnitude', fontsize=18)

posterior_samples_sed[:,1] = posterior_samples_sed[:,1] - 9
posterior_samples_im[:,1] = posterior_samples_im[:,1] - 9
posterior_samples_combined[:,1] = posterior_samples_combined[:,1] - 9  


fig = corner.corner(posterior_samples_sed, truths=x_truth, color='dodgerblue', range=ranges, labels=labels, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k', color='firebrick', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_combined, truths=x_truth, truth_color='k', color='green', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])

sed_legend_patch = mpatches.Patch(color='dodgerblue', label='Photometry')
image_legend_patch = mpatches.Patch(color='firebrick', label='Image')
combined_legend_patch = mpatches.Patch(color='green', label='Combined')
fig.legend(handles=[sed_legend_patch, image_legend_patch, combined_legend_patch],
           loc='upper right', fontsize='18', title='Posterior Source', frameon=True, title_fontsize='18')
fig.set_size_inches(12, 12)
for label in fig.get_axes():
    label.xaxis.label.set_size(18)
    label.yaxis.label.set_size(18)
    for tick in label.get_xticklabels():
        tick.set_fontsize(14)
    for tick in label.get_yticklabels():
        tick.set_fontsize(14)
plt.show()

# Figure 4: 5-Band Photometry vs. Image vs. Combined $1\sigma$ Credible Interval

In [ ]:
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

# Match coordinate length to sample array length
n_plot_5b = len(primary_individual_scatters["5 Band Photometry"])
# x_m_5b = mvals_full if n_plot_5b == len(mvals_full) else mvals
x_m_5b = mvals_full
# y_s_5b = sfrvals_full if n_plot_5b == len(sfrvals_full) else sfrvals
y_s_5b = sfrvals_full
vit_scatters_5b = primary_individual_scatters["Finetuned ViT"] if len(primary_individual_scatters["Finetuned ViT"]) == n_plot_5b else primary_individual_scatters["Finetuned ViT"][train_index:]

plt.subplots(figsize=(15, 21), nrows=6, ncols=3)
plt.suptitle('5 Band Photometry vs Image vs Combined 1$\sigma$ Credible Interval', fontsize=18, y=1)
for qty in range(6):
    c_lower_bound = np.nanpercentile(vit_scatters_5b[:, qty], 2)
    c_upper_bound = np.nanpercentile(vit_scatters_5b[:, qty], 98)
    
    plt.subplot(6, 3, qty*3+1)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=primary_individual_scatters["5 Band Photometry"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Photometry}$', fontsize=18)
    
    plt.subplot(6, 3, qty*3+2)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=vit_scatters_5b[:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$  [$M_{\odot}$]', fontsize=18)  
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Image}$', fontsize=18)
    plt.tight_layout()

    plt.subplot(6, 3, qty*3+3)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=primary_individual_scatters["Combined (ViT + Bands)"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$  [$M_{\odot}$]', fontsize=18) 
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Photometry+Image}$', fontsize=18)

plt.show()

# Figure 5: Population-Level Degeneracies ($t_{50}$, $A_v$, $12+\log(\mathrm{O/H})$ Residuals)

This runs a quick version of the plot that retuns the right rho values, but assumes the posterior is Gaussian. 

The corner plot produced is overly smooth, but making the real plot requires a lot of RAM to load up all samples from the 2 models (this version is used in the paper and is found in the commented out cell below)

In [ ]:
import matplotlib.lines as mlines
import numpy as np
import matplotlib.pyplot as plt
import corner

# 1. Calculate median standard deviation for each parameter (sigma = (p84 - p16) / 2)
# (Assuming 'im_p16' and 'im_p84' match the 'covs_image' variables you used)
std_bands = np.nanmedian((sed5_p84[:, 0:6] - sed5_p16[:, 0:6]) / 2.0, axis=0)
std_combined = np.nanmedian((im_p84[:, 0:6] - im_p16[:, 0:6]) / 2.0, axis=0)

# 2. Extract the offset 5x5 Spearman correlation matrices (rho)
rho_bands = np.nanmedian(covs_5_band, axis=0)
rho_combined = np.nanmedian(covs_image, axis=0)

# 3. Build true 6x6 symmetric covariance matrices: Cov(X, Y) = rho * std(X) * std(Y)
def build_true_cov_matrix(std_arr, rho_matrix):
    cov = np.zeros((6, 6))
    for i in range(6):
        cov[i, i] = std_arr[i] ** 2  # Variance on the diagonal
        for j in range(i):
            r = rho_matrix[i-1, j]   # Shift index back to match your 5x5 matrix
            cov[i, j] = r * std_arr[i] * std_arr[j]
            cov[j, i] = cov[i, j]
    return cov

true_cov_bands = build_true_cov_matrix(std_bands, rho_bands)
true_cov_combined = build_true_cov_matrix(std_combined, rho_combined)

# 4. Setup plotting parameters for [t50, Av, 12+log(O/H)]
indices_to_plot = [3, 4, 5]
n_params_subset = len(indices_to_plot)

# Extract the specific 3x3 covariance sub-matrices for the parameters we are plotting
cov_subset_bands = true_cov_bands[np.ix_(indices_to_plot, indices_to_plot)]
cov_subset_combined = true_cov_combined[np.ix_(indices_to_plot, indices_to_plot)]

# 5. Generate synthetic pooled residuals from the summary statistics
n_mock_samples = 150000
subset_bands = np.random.multivariate_normal(mean=np.zeros(n_params_subset), cov=cov_subset_bands, size=n_mock_samples)
subset_combined = np.random.multivariate_normal(mean=np.zeros(n_params_subset), cov=cov_subset_combined, size=n_mock_samples)

# 6. Setup plotting styling
subset_labels = [r'$\Delta$ t$_{50}$', r'$\Delta$ Av', r'$\Delta$ 12+log(O/H)'] 
color_bands = 'dodgerblue'
color_combined = 'red'
plot_range = [(-6, 6), (-2.5, 2.5), (-1.5, 1.5)] 

kwargs_bands = {
    "labels": subset_labels,
    "label_kwargs": {"fontsize": 18},
    "hist_kwargs": {"density": True, "linewidth": 2},
    "plot_datapoints": False,
    "plot_density": True,
    "fill_contours": True,
    "alpha": 0.3,                 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,                   
}

kwargs_combined = {
    "labels": subset_labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5}, 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,
}

# 7. Generate the corner plot
fig = plt.figure(figsize=(12, 12))
fig = corner.corner(subset_bands, fig=fig, color=color_bands, **kwargs_bands)
corner.corner(subset_combined, fig=fig, color=color_combined, **kwargs_combined)

axes = np.array(fig.axes).reshape((n_params_subset, n_params_subset))
bbox_props = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8)

for yi in range(n_params_subset):
    for xi in range(yi):
        ax = axes[yi, xi]
        ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
        orig_y = indices_to_plot[yi]
        orig_x = indices_to_plot[xi]
        
        # Read the rho value directly from the offset 5x5 Spearman matrices
        rho_val_bands = rho_bands[orig_y - 1, orig_x]
        rho_val_comb = rho_combined[orig_y - 1, orig_x]
        
        txt_bands = f"$\\rho_{{phot}} = {rho_val_bands:.2f}$"
        txt_comb = f"$\\rho_{{img}} = {rho_val_comb:.2f}$"
        
        ax.text(0.05, 0.96, txt_bands, transform=ax.transAxes, 
                fontsize=16, color=color_bands, ha='left', va='top', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.86, txt_comb, transform=ax.transAxes, 
                fontsize=16, color=color_combined, ha='left', va='top', weight='bold', bbox=bbox_props)
    
    axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

legend_handles = [
    mlines.Line2D([], [], color=color_bands, lw=4, label='5 Band Photometry'),
    mlines.Line2D([], [], color=color_combined, lw=3, label='Image')
]

# Fixed broken newline on the legend call
fig.axes[n_params_subset - 1].legend(handles=legend_handles, loc='center left', fontsize=18, frameon=False)
plt.suptitle("Population-Level Degeneracies (Residuals)", fontsize=22, y=1.02)
plt.show()

In [ ]:
# import matplotlib.lines as mlines

# FAST_MODE = True  
# STRIDE = 100       

# def get_pooled_residuals(samples_array):
#     medians = np.nanmedian(samples_array, axis=1, keepdims=True)
#     residuals = samples_array - medians
#     return residuals.reshape(-1, samples_array.shape[-1])

# # 1. Compute pooled residuals on the 6 physical parameters ([:, :, 0:6])
# subset_bands = get_pooled_residuals(all_samples_5_band[:, :, 0:6])
# subset_combined = get_pooled_residuals(all_samples_image[:, :, 0:6])

# if FAST_MODE:
#     print(f"Running in FAST_MODE. Plotting 1/{STRIDE} of the samples...")
#     subset_bands = subset_bands[::STRIDE]
#     subset_combined = subset_combined[::STRIDE]

# # 2. Extract Pre-calculated Median Covariances for the 6 physical parameters ([:, 0:5, 0:5])
# # median_cov_bands = np.nanmedian(all_covs_5_band[:, 0:5, 0:5], axis=0)
# # median_cov_combined = np.nanmedian(all_covs_image[:, 0:5, 0:5], axis=0)

# median_cov_bands = np.nanmedian(covs_5_band[:, 0:5, 0:5], axis=0)
# median_cov_combined = np.nanmedian(covs_image[:, 0:5, 0:5], axis=0)


# # 3. Setup plotting parameters for [t50, Av, 12+log(O/H)]
# indices_to_plot = [3, 4, 5]
# n_params_subset = len(indices_to_plot)

# subset_bands = subset_bands[:, indices_to_plot]
# subset_combined = subset_combined[:, indices_to_plot]

# subset_labels = [r'$\Delta$ t$_{50}$', r'$\Delta$ Av', r'$\Delta$ 12+log(O/H)'] 
# color_bands = 'dodgerblue'
# color_combined = 'red'
# plot_range = [(-6, 6), (-2.5, 2.5), (-1.5, 1.5)] 

# kwargs_bands = {
#     "labels": subset_labels,
#     "label_kwargs": {"fontsize": 18},
#     "hist_kwargs": {"density": True, "linewidth": 2},
#     "plot_datapoints": False,
#     "plot_density": True,
#     "fill_contours": True,
#     "alpha": 0.3,                 
#     "levels": (0.68, 0.95),
#     "range": plot_range,          
#     "bins": 30,                   
# }

# kwargs_combined = {
#     "labels": subset_labels,
#     "hist_kwargs": {"density": True, "linewidth": 2.5},
#     "plot_datapoints": False,
#     "plot_density": False,        
#     "fill_contours": False,       
#     "contour_kwargs": {"linewidths": 2.5}, 
#     "levels": (0.68, 0.95),
#     "range": plot_range,          
#     "bins": 30,
# }

# # 4. Generate the corner plot
# fig = plt.figure(figsize=(12, 12))
# fig = corner.corner(subset_bands, fig=fig, color=color_bands, **kwargs_bands)
# corner.corner(subset_combined, fig=fig, color=color_combined, **kwargs_combined)

# axes = np.array(fig.axes).reshape((n_params_subset, n_params_subset))
# bbox_props = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8)

# for yi in range(n_params_subset):
#     for xi in range(yi):
#         ax = axes[yi, xi]
#         ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
#         ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
#         ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
#         orig_y = indices_to_plot[yi]
#         orig_x = indices_to_plot[xi]
        
#         rho_bands = median_cov_bands[orig_y - 1, orig_x]
#         rho_comb = median_cov_combined[orig_y - 1, orig_x]
        
#         txt_bands = f"$\\rho_{{phot}} = {rho_bands:.2f}$"
#         txt_comb = f"$\\rho_{{img}} = {rho_comb:.2f}$"
        
#         ax.text(0.05, 0.96, txt_bands, transform=ax.transAxes, 
#                 fontsize=16, color=color_bands, ha='left', va='top', weight='bold', bbox=bbox_props)
#         ax.text(0.05, 0.86, txt_comb, transform=ax.transAxes, 
#                 fontsize=16, color=color_combined, ha='left', va='top', weight='bold', bbox=bbox_props)
    
#     axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

# legend_handles = [
#     mlines.Line2D([], [], color=color_bands, lw=4, label='5 Band Photometry'),
#     mlines.Line2D([], [], color=color_combined, lw=3, label='Image')
# ]
# fig.axes[n_params_subset - 1].legend(handles=legend_handles, loc='center left', fontsize=18, frameon=False)
# plt.suptitle("Population-Level Degeneracies (Residuals)", fontsize=22, y=1.02)
# plt.show()

# Figure 6: 10 Gyr SFH Reconstruction Across $M_*$-sSFR Bins
In the 34-dimensional target array `theta_full_sfh_sfr`, index `1` corresponds to present-day $\log\mathrm{SFR}$, indices `6` and `7` are $t_{10}$ and $t_{90}$, and indices `8` through `33` correspond to $\log\mathrm{SFR}$ at the 26 lookback times from $0.1$ to $10.0\,\mathrm{Gyr}$.

In [ ]:
# Mapping of lookback time to column index in the 35D sample arrays

time_to_idx_35d = {
    0.0: 8, 0.1: 9, 0.3: 10, 0.5: 11, 0.7: 12, 0.8: 13, 1.0: 14,
    1.3: 15, 1.5: 16, 1.7: 17, 2.0: 18, 2.5: 19, 3.0: 20,
    3.5: 21, 4.0: 22, 4.5: 23, 5.0: 24, 5.5: 25, 6.0: 26,
    6.5: 27, 7.0: 28, 7.5: 29, 8.0: 30, 8.5: 31, 9.0: 32,
    9.5: 33, 10.0: 34
}

timescale_indices_35d = {
    ("today" if t == 0.0 else f"{t}gyr"): idx for t, idx in time_to_idx_35d.items()
}

# Build dictionary of median predicted log SFR for each lookback time from the precomputed image summaries
n_im_samples = len(im_p50)
slice_im = slice(None) if n_im_samples == len(mvals_full) else slice(train_index, None)

predicted_timescales_image = {}
for t, col_idx in time_to_idx_35d.items():
    key = "now" if t == 0.0 else f"{t}gyr"
    t_key = "now" if t == 0.0 else t
    predicted_timescales_image[key] = (
        timescales_data_cami_fixed[t_key]['mass'][slice_im],
        im_p50[:, col_idx]  # Directly slice the precomputed median instead of calculating it
    )

In [ ]:
gswlc_sfh_load = np.load('SFH_FILE_DIR_HERE')
gswlc_sfh_mod = np.array([gswlc_sfh_load['sfhs'], gswlc_sfh_load['timeax']]).transpose(1, 0, 2) # Shape: (N_galaxies, 2, N_timepoints)

In [ ]:
reconstruction_times = [
    0.0, 0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 1.7, 2.0,
    2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 
    7.5, 8.0, 8.5, 9.0, 9.5, 10.0
]

mass_bins = [9.0, 10.2, 10.8, 12.0]
mass_labels = ["Low Mass (<10.2)", "Med Mass (10.2-10.8)", "High Mass (>10.8)"]

ssfr_bins = [-13.0, -11.5, -10.5, -9.5, -7.0] 
ssfr_labels = [
    "Quiescent (sSFR < -11.5)", 
    "Green Valley(-11.5 < sSFR < -10.5)", 
    "Main Sequence (-10.5 < sSFR < -9.5)", 
    "Starburst (sSFR > -9.5)"
]

gal_mass = mvals_full[slice_im]
gal_sfr  = sfrvals_full[slice_im]
# gal_sfr = theta_full_sfh_sfr[slice_im, 1]  # Use the precomputed SFH SFR values
gal_ssfr = gal_sfr - gal_mass
sfh_mod_sub = gswlc_sfh_mod[slice_im]

fig, axes = plt.subplots(4, 3, figsize=(18, 16), sharex=True, sharey=False)
common_lookback_grid = np.linspace(0, 10.0, 200)

print("Generating 10 Gyr Reconstruction Grid (4 Rows: SB, MS, GV, Q)...")

for row_idx, ssfr_idx in enumerate([3, 2, 1, 0]): 
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        
        m_min, m_max = mass_bins[col_idx], mass_bins[col_idx+1]
        s_min, s_max = ssfr_bins[ssfr_idx], ssfr_bins[ssfr_idx+1]
        
        bin_mask = (gal_mass >= m_min) & (gal_mass < m_max) & \
                   (gal_ssfr >= s_min) & (gal_ssfr < s_max)
        
        indices = np.where(bin_mask)[0]
        n_count = len(indices)
        bin_title = f"{mass_labels[col_idx]}\n{ssfr_labels[ssfr_idx]}\n(N={n_count})"

        if n_count == 0:
            ax.text(0.5, 0.5, "No Galaxies", ha='center', transform=ax.transAxes)
            ax.set_title(bin_title, fontsize=10)
            continue

        true_sfhs_list = []
        pred_sfhs_list = []
        
        for idx in indices:
            sfr_true = sfh_mod_sub[idx, 0, :]  
            t_cosmic = sfh_mod_sub[idx, 1, :]
            t_obs = np.max(t_cosmic) 
            t_lookback_gal = t_obs - t_cosmic
            
            sort_order = np.argsort(t_lookback_gal)
            sfr_interp = np.interp(common_lookback_grid, 
                                   t_lookback_gal[sort_order], 
                                   sfr_true[sort_order])
            true_sfhs_list.append(sfr_interp)
            
            preds = []
            for t in reconstruction_times:
                key = 'now' if t == 0.0 else f'{t}gyr'
                preds.append(predicted_timescales_image[key][1][idx])
            pred_sfhs_list.append(preds)

        true_arr = np.array(true_sfhs_list)
        pred_arr = np.array(pred_sfhs_list)
            
        true_med = np.nanmedian(true_arr, axis=0)
        true_p16 = np.nanpercentile(true_arr, 16, axis=0)
        true_p84 = np.nanpercentile(true_arr, 84, axis=0)
        
        pred_med = np.nanmedian(pred_arr, axis=0)
        pred_p16 = np.nanpercentile(pred_arr, 16, axis=0)
        pred_p84 = np.nanpercentile(pred_arr, 84, axis=0)

        ax.fill_between(common_lookback_grid, true_p16, true_p84, 
                        color='black', alpha=0.1, label='True Spread')
        ax.plot(common_lookback_grid, true_med, color='black', lw=2, label='Avg Truth')
        
        ax.fill_between(reconstruction_times, 10**pred_p16, 10**pred_p84, 
                        color='red', alpha=0.2, label='Model Spread')
        ax.plot(reconstruction_times, 10**pred_med, color='red', 
                linestyle='--', marker='o', markersize=3, label='Avg Model')

        ax.set_title(bin_title, fontsize=14)
        ax.set_xlim(-0.2, 10.2)

        if col_idx == 0: ax.set_ylim(0, 3)
        if col_idx == 1: ax.set_ylim(0, 12)
        if col_idx == 2: ax.set_ylim(0, 30)

        ax.grid(alpha=0.3)
        if row_idx == 0 and col_idx == 0:
            ax.legend(loc='upper left', fontsize=14)

fig.supxlabel("Lookback Time [Gyr] (0 = Today)", fontsize=16)
fig.supylabel("SFR [$M_\odot$/yr]", fontsize=16)
plt.tight_layout()
plt.show()

# Figure 7: Continuously Accurate SFH Prediction (Continuous Morphological Memory)

In [ ]:
def compute_sfh_memory_dicts(p16_34d, p50_34d, p84_34d, truth_34d):
    """
    Computes individual, cumulative (continuous), and step-failure accuracy dictionaries
    across the 27 SFH lookback timesteps using precomputed percentiles.
    """
    individual_acc = {}
    cumulative_acc = {}
    failed_step = {}
    current_intersection = None
    threshold = 0.15  # Define the threshold for accuracy

    print(f"{'Time':<8} | {'Indiv Acc':<10} | {'Cumulative':<10} | {'Failed Here'}")
    print("-" * 45)

    for label, col_idx in timescale_indices_34d.items():
        # Slice directly from the precomputed summary arrays
        p16_raw = p16_34d[:, col_idx]
        p84_raw = p84_34d[:, col_idx]
        truth = truth_34d[:, col_idx]
        
        
        with np.errstate(invalid='ignore'):
            p16 = p16_raw.copy()
            p84 = p84_raw.copy()

            p16[np.abs(p50_34d[:, col_idx] - p16_raw) <= threshold] = -threshold + p50_34d[:, col_idx][np.abs(p50_34d[:, col_idx] - p16_raw) <= threshold]
            p84[np.abs(p84_raw - p50_34d[:, col_idx]) <= threshold] = threshold + p50_34d[:, col_idx][np.abs(p84_raw - p50_34d[:, col_idx]) <= threshold]

            is_accurate_mask = (truth >= p16) & (truth <= p84)
            
        acc_indices = np.where(is_accurate_mask)[0]
        individual_acc[label] = acc_indices

        if current_intersection is None:
            current_intersection = acc_indices
            failed_step[label] = np.array([])
        else:
            failures = np.setdiff1d(current_intersection, acc_indices)
            failed_step[label] = failures
            current_intersection = np.intersect1d(current_intersection, acc_indices)
            
        cumulative_acc[label] = current_intersection
        print(f"{label:<8} | {len(acc_indices):<10} | {len(current_intersection):<10} | {len(failed_step[label])}")
        
    return individual_acc, cumulative_acc, failed_step

# truth_im_34d = theta_full_sfh_sfr if len(all_samples_image) == len(theta_full_sfh_sfr) else theta_full_sfh_sfr[train_index:]
individual_accurate_cami, cumulative_accurate_cami, failed_at_step_cami = compute_sfh_memory_dicts(im_p16, im_p50,im_p84, theta_full_sfh_sfr)

In [ ]:
# Construct Continuous Memory Depth Array & Plot Figure 7
timescale_map = [
    (0.0, "today"), (0.1, "0.1gyr"), (0.3, "0.3gyr"), (0.5, "0.5gyr"),
    (0.7, "0.7gyr"), (0.8, "0.8gyr"), (1.0, "1.0gyr"), (1.3, "1.3gyr"),
    (1.5, "1.5gyr"), (1.7, "1.7gyr"), (2.0, "2.0gyr"), (2.5, "2.5gyr"),
    (3.0, "3.0gyr"), (3.5, "3.5gyr"), (4.0, "4.0gyr"), (4.5, "4.5gyr"),
    (5.0, "5.0gyr"), (5.5, "5.5gyr"), (6.0, "6.0gyr"), (6.5, "6.5gyr"),
    (7.0, "7.0gyr"), (7.5, "7.5gyr"), (8.0, "8.0gyr"), (8.5, "8.5gyr"),
    (9.0, "9.0gyr"), (9.5, "9.5gyr"), (10.0, "10.0gyr")
]

x_mass_im = mvals_full 
y_sfr_im  = sfrvals_full 

memory_depth = np.zeros(len(x_mass_im))
for t_val, key in timescale_map:
    if key in cumulative_accurate_cami:
        memory_depth[cumulative_accurate_cami[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_mass_im, y_sfr_im, C=memory_depth, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=6, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
plt.show()

In [ ]:
# Correlation check between continuous morphological memory and morphological summary statistics (Sec. 4.3)
morph_slice = slice(None) if len(memory_depth) == len(gswlc_data_merged) else slice(train_index, None)
# morph_slice = small_dif_mask
for col_name in ['bulge_n_morph', 'ttype_morph', 'bt_morph', 'pbulge_morph']:
    y_pred = np.array(gswlc_data_merged[col_name])[morph_slice]
    mask = ~np.isnan(memory_depth) & ~np.isnan(y_pred)
    rho, p_val = spearmanr(memory_depth[mask], y_pred[mask])
    print(f"{col_name:<15} | Spearman rho: {rho:>7.4f} | p-value: {p_val:.4e}")

# Figure 8: UMAP Projection of the ViT Image Embedding Space

In [ ]:
import umap
import matplotlib.gridspec as gridspec

data_embeds = np.array(gswlc_finetuned_embeds)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean')
embedding = reducer.fit_transform(data_embeds)

In [ ]:
import numpy as np
from astropy.cosmology import Planck18
import astropy.units as u

# Define your reference cosmology (e.g., Planck 2018 or WMAP9)
cosmo = Planck18 

# Assuming you have 1D numpy arrays or pandas Series:
theta_arcsec = gswlc_data_merged['half_light_radius_morph']
z_spec       = gswlc_data_merged['specz']

# Compute angular diameter distance in kpc
# astropy's kpc_proper_per_arcmin gives a direct scale conversion:
kpc_per_arcsec = cosmo.kpc_proper_per_arcmin(z_spec).to(u.kpc / u.arcsec).value

# Convert radius to physical kpc
r_half_kpc = theta_arcsec * kpc_per_arcsec
r_half_kpc = np.array(r_half_kpc)

In [ ]:
# Figure 8 (Lower Panels): 2x4 Grid Colored by Morphological, Physical, and Precision Metrics
fig, axes = plt.subplots(3, 4, figsize=(28, 15), sharex=True, sharey=True)

axis_label_fontsize = 18
cbar_label_fontsize = 18
cbar_tick_fontsize = 14 

fig.supxlabel("UMAP Dimension 1", fontsize=axis_label_fontsize)
fig.supylabel("UMAP Dimension 2", fontsize=axis_label_fontsize, x=-0.01)


# --- Subplot 1 (Top, Col 1): Number Density ---
ax1 = axes[0, 0]
hb1 = ax1.hexbin(embedding[:, 0], embedding[:, 1], gridsize=40, cmap='Blues', mincnt=1)
cbar1 = fig.colorbar(hb1, ax=ax1)
cbar1.set_label('Number Density', fontsize=cbar_label_fontsize)
cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 2 (Top, Col 2): Continuous Accurate SFH Prediction ---
ax2 = axes[0, 1]
emb_im = embedding if len(memory_depth) == len(embedding) else embedding[train_index:]
data2 = memory_depth
hb2 = ax2.hexbin(emb_im[:, 0], emb_im[:, 1], C=data2, gridsize=40, cmap='Spectral_r',
                 vmin=0, vmax=6)
cbar2 = fig.colorbar(hb2, ax=ax2, extend='both')
cbar2.set_label('Continuous Accurate SFH Prediction', fontsize=cbar_label_fontsize)
cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 3 (Top, Col 3): sSFR Inferred Posterior Width ---
ax3 = axes[0, 2]
data3 = primary_individual_scatters["Finetuned ViT"][:, 1]
hb3 = ax3.hexbin(emb_im[:, 0], emb_im[:, 1], C=data3, gridsize=40, cmap='viridis',
                 vmin=np.nanpercentile(data3, 2), vmax=np.nanpercentile(data3, 98))
cbar3 = fig.colorbar(hb3, ax=ax3,extend='both')
cbar3.set_label('sSFR Inferred Posterior Width', fontsize=cbar_label_fontsize)
cbar3.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 4 (Top, Col 4): sSFR Prediction Residuals ---
ax4 = axes[0, 3]
emb_5b = embedding if len(allsederr_combined_5_bands_finetuned_vit_full_new) == len(embedding) else embedding[train_index:]
data4 = allsederr_combined_5_bands_finetuned_vit_full_new[:, 1]
hb4 = ax4.hexbin(emb_5b[:, 0], emb_5b[:, 1], C=data4, gridsize=40, cmap='viridis',
                vmin=np.nanpercentile(data3, 2), vmax=np.nanpercentile(data3, 98))
cbar4 = fig.colorbar(hb4, ax=ax4,extend='both')
cbar4.set_label('sSFR Prediction Residuals', fontsize=cbar_label_fontsize)
cbar4.ax.tick_params(labelsize=cbar_tick_fontsize)

morph_colour_way = 'vanimo'
# --- Subplot 5 (Middle, Col 1): Prominent Bulge ---
ax5 = axes[1, 0]
data5 = gswlc_data_merged['pbulge_morph']
hb5 = ax5.hexbin(embedding[:, 0], embedding[:, 1], C=data5, gridsize=40, cmap=morph_colour_way,
                 vmin=np.nanpercentile(data5, 2), vmax=np.nanpercentile(data5, 98))
cbar5 = fig.colorbar(hb5, ax=ax5,extend='both')
cbar5.set_label('Prominent Bulge', fontsize=cbar_label_fontsize)
cbar5.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 6 (Middle, Col 2): T-Type ---
ax6 = axes[1, 1]
data6 = gswlc_data_merged['ttype_morph']
hb6 = ax6.hexbin(embedding[:, 0], embedding[:, 1], C=data6, gridsize=40, cmap=morph_colour_way,
                 vmin=np.nanpercentile(data6, 2), vmax=np.nanpercentile(data6, 98))
cbar6 = fig.colorbar(hb6, ax=ax6,extend='both')
cbar6.set_label('T-Type', fontsize=cbar_label_fontsize)
cbar6.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 7 (Middle, Col 3): T-Type ---
ax7 = axes[1, 2]
data7 = np.log10(r_half_kpc)
hb7 = ax7.hexbin(embedding[:, 0], embedding[:, 1], C=data7, gridsize=40, cmap=morph_colour_way,
                 vmin=np.nanpercentile(data7, 2), vmax=np.nanpercentile(data7, 98))
cbar7 = fig.colorbar(hb7, ax=ax7,extend='both')
cbar7.set_label('Log Half-light Radius (kpc)', fontsize=cbar_label_fontsize)
cbar7.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 8 (Middle, Col 4): T-Type ---
ax8 = axes[1, 3]
data8 = gswlc_data_merged['bulge_n_morph']
hb8 = ax8.hexbin(embedding[:, 0], embedding[:, 1], C=data8, gridsize=40, cmap=morph_colour_way,
                 vmin=1, vmax=5)
cbar8 = fig.colorbar(hb8, ax=ax8,extend='both')
cbar8.set_label('Sersic Index', fontsize=cbar_label_fontsize)
cbar8.ax.tick_params(labelsize=cbar_tick_fontsize)

physical_param_colour = 'berlin'
# --- Subplot 9 (Bottom, Col 1): Mass ---
ax9 = axes[2, 0]
# data4 = gswlc_data_merged['logSSFR_50'] - 9
data9 = gswlc_data['logM_50_x']
hb9 = ax9.hexbin(embedding[:, 0], embedding[:, 1], C=data9, gridsize=40, cmap=physical_param_colour,
                 vmin=np.nanpercentile(data9, 2), vmax=np.nanpercentile(data9, 98))
cbar9 = fig.colorbar(hb9, ax=ax9,extend='both')
cbar9.set_label('log $M_*$', fontsize=cbar_label_fontsize)
cbar9.ax.tick_params(labelsize=cbar_tick_fontsize)


# --- Subplot 10 (Bottom, Col 2): sSFR ---
ax10 = axes[2, 1]
# data4 = gswlc_data_merged['logSSFR_50'] - 9
data10 = tempssfr -9
hb10 = ax10.hexbin(embedding[:, 0], embedding[:, 1], C=data10, gridsize=40, cmap=physical_param_colour,
                 vmin=np.nanpercentile(data10, 2), vmax=np.nanpercentile(data10, 98))
cbar10 = fig.colorbar(hb10, ax=ax10,extend='both')
cbar10.set_label('log sSFR', fontsize=cbar_label_fontsize)
cbar10.ax.tick_params(labelsize=cbar_tick_fontsize)


# --- Subplot 11 (Bottom, Col 3): Av ---
ax11 = axes[2, 2]
data11 = gswlc_data_merged['tauV_50']
hb11 = ax11.hexbin(embedding[:, 0], embedding[:, 1], C=data11, gridsize=40, cmap=physical_param_colour,
                 vmin=np.nanpercentile(data11, 2), vmax=np.nanpercentile(data11, 98))
cbar11 = fig.colorbar(hb11, ax=ax11,extend='both')
cbar11.set_label('Av', fontsize=cbar_label_fontsize)
cbar11.ax.tick_params(labelsize=cbar_tick_fontsize)


# --- Subplot 4 (Bottom, Col 4): t50
ax12 = axes[2, 3]
# data4 = gswlc_data_merged['logSSFR_50'] - 9
data12 = gswlc_data['t_50']
hb12 = ax12.hexbin(embedding[:, 0], embedding[:, 1], C=data12, gridsize=40, cmap=physical_param_colour,
                 vmin=np.nanpercentile(data12, 2), vmax=np.nanpercentile(data12, 98))
cbar12 = fig.colorbar(hb12, ax=ax12,extend='both')
cbar12.set_label('t50', fontsize=cbar_label_fontsize)
cbar12.ax.tick_params(labelsize=cbar_tick_fontsize)


plt.tight_layout()  
plt.show()

In [ ]:
# Figure 8 (Top-Left Panel): Galaxy Image Cutouts Mapped Across UMAP Space
def umap_plotter_images(embeds, images, ranges=[-8, 25, -15, 21], nplt=40, dist_threshold=0.3):
    fig = plt.figure(figsize=(nplt*2, nplt*2))
    spec2 = gridspec.GridSpec(ncols=nplt, nrows=nplt, figure=fig, hspace=0, wspace=0)
    for ii, i in enumerate(np.linspace(ranges[0], ranges[1], nplt)):
        for jj, j in enumerate(np.linspace(ranges[2], ranges[3], nplt)):
            dij = np.argmin(np.abs(embeds[:, 0] - i) + np.abs(embeds[:, 1] - j))
            dist = (np.abs(embeds[dij, 0] - i) + np.abs(embeds[dij, 1] - j))
            if dist < dist_threshold:
                sel_image = Image.open(io.BytesIO(images[dij]))
                fig.add_subplot(spec2[nplt-1-jj, ii]).imshow(sel_image)
                plt.axis('off')
    # plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/umap_vit_embeddings_finetuned_images.pdf', dpi=300, bbox_inches='tight')            
    plt.show()

umap_plotter_images(embedding, gswlc_images_skyserver, nplt=40, dist_threshold=0.3)

In [ ]:
# Figure 8 (Top-Right Panel): Median SFH Trajectories Mapped Across UMAP Space
nsfh = 7

def get_t50_color(t50):
    t50_scaled = min((t50) / 13.25, 1.0)
    return plt.cm.berlin(t50_scaled)

def get_sfh(gal_id):
    sfh = gswlc_sfh_mod[gal_id][0]
    timeax = gswlc_sfh_mod[gal_id][1]
    ssfr = gswlc_data['logSSFR_50'][gal_id] - 9
    return sfh, timeax, ssfr, gswlc_data['t_50'][gal_id]

def umap_plotter_sfh(embeds, ranges=[-8, 25, -15, 21], nplt=40, dist_threshold=0.3):
    fig = plt.figure(figsize=(nplt*2, nplt*2))
    spec2 = gridspec.GridSpec(ncols=nplt, nrows=nplt, figure=fig, hspace=0, wspace=0)
    for ii, i in tqdm(enumerate(np.linspace(ranges[0], ranges[1], nplt))):
        for jj, j in enumerate(np.linspace(ranges[2], ranges[3], nplt)):
            dijs = np.argsort(np.abs(embeds[:, 0] - i) + np.abs(embeds[:, 1] - j))[0:nsfh]
            dist = (np.abs(embeds[dijs[-1], 0] - i) + np.abs(embeds[dijs[-1], 1] - j))
            if dist < dist_threshold:
                sfhs, times, ssfrs, t50s = [], [], [], []
                for npi in range(nsfh):
                    a, b, c, d = get_sfh(dijs[npi])
                    sfhs.append(a / np.sum(a)), times.append(b), ssfrs.append(c), t50s.append(d)
                    
                splined_time = np.linspace(0, np.amax(times), 1000)
                splined_sfhs = [np.interp(splined_time, times[k], np.flip(sfhs[k], 0) / np.amax(sfhs[k])) for k in range(len(sfhs))]
                fig.add_subplot(spec2[nplt-1-jj, ii]).plot(splined_time, np.nanmedian(splined_sfhs, 0), lw=5, color=get_t50_color(np.nanmedian(t50s)))
                plt.fill_between(splined_time, np.nanpercentile(splined_sfhs, 16, 0), np.nanpercentile(splined_sfhs, 84, 0), color=get_t50_color(np.nanmedian(t50s)), alpha=0.1)
                plt.ylim(0, 1)
                plt.axis('off')
    # plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/umap_vit_embeddings_finetuned_sfh.pdf', dpi=300, bbox_inches='tight')
    plt.show()

umap_plotter_sfh(embedding, nplt=40)

# Figure 9: Toy Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d, gaussian_filter1d

# ==============================================================================
# 1. EXTRACT REAL DATA SLICE AT FIXED STELLAR MASS
# ==============================================================================
# Define fixed stellar mass slice (e.g., 10.3 <= log M* <= 10.7, centered at 10.5)
fixed_log_mass = 10.5
mass_window = 0.2  # +/- 0.2 dex around fixed_log_mass

t_lb_bins = np.array([
    0.0, 0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 1.7, 2.0,
    2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0,
    7.5, 8.0, 8.5, 9.0, 9.5, 10.0
])
col_idx_34d = [1] + list(range(8, 34))

# Ensure continuous memory (`memory_depth`) is computed for the real dataset
n_im = len(memory_depth)
slice_im = slice(None) if n_im == len(mvals_full) else slice(train_index, None)
real_mass = mvals_full[slice_im]
# real_sfr  = sfrvals_full[slice_im]
real_sfr = tempssfr + theta[:,0] - 9
real_ssfr = real_sfr - real_mass

if 'memory_depth' not in globals() or len(memory_depth) != n_im:
    print("Computing continuous memory depth from `all_samples_image`...")
    truth_34d = theta_full_sfh_sfr[slice_im]
    is_acc_real = np.zeros((n_im, len(t_lb_bins)), dtype=bool)
    for k, c_idx in enumerate(col_idx_34d):
        # p16 = np.nanpercentile(all_samples_image[:, :, c_idx], 16, axis=1)
        p16 = im_p16[:, c_idx]  # Use precomputed 16th percentile
        # p84 = np.nanpercentile(all_samples_image[:, :, c_idx], 84, axis=1)
        p84 = im_p84[:, c_idx]  # Use precomputed 84th percentile
        is_acc_real[:, k] = (truth_34d[:, c_idx] >= p16) & (truth_34d[:, c_idx] <= p84)
    
    cum_acc_real = np.minimum.accumulate(is_acc_real, axis=1)
    memory_depth = np.zeros(n_im)
    for k, t_val in enumerate(t_lb_bins):
        memory_depth[cum_acc_real[:, k]] = t_val

# Filter real galaxies within the fixed mass slice
mass_mask_real = (real_mass >= fixed_log_mass - mass_window) & (real_mass <= fixed_log_mass + mass_window)
ssfr_real_slice = real_ssfr[mass_mask_real]
mem_real_slice  = memory_depth[mass_mask_real]
print(f"Selected N = {np.sum(mass_mask_real)} real galaxies in {fixed_log_mass - mass_window:.1f} <= log M* <= {fixed_log_mass + mass_window:.1f}")

# ==============================================================================
# 2. TWO-TIMESCALE TOY MODEL (t2 >> t1 vs. t1 ~= t2) AT FIXED MASS
# ==============================================================================
def simulate_sam_fixed_mass(regime='lag', N_sim=15000, fixed_log_mass=10.5,
                            t1=0.2, t2=2.0, seed=42):
    rng = np.random.default_rng(seed)
    dt = 0.02
    t_cosmic = np.arange(0.0, 10.0 + dt, dt)
    t_lb_hr = 10.0 - t_cosmic
    bin_idx = [np.argmin(np.abs(t_lb_hr - tb)) for tb in t_lb_bins]
    
    # Slightly shift MS center (-0.12 dex) to align with GSWLC median MS peak at log M* = 10.5
    log_sfr_ms = (0.83 * fixed_log_mass - 8.33) - 0.12
    sfr_ms_linear = 10**log_sfr_ms
    
    pops = rng.choice(['Q', 'GV', 'MS', 'SB'], size=N_sim, p=[0.28, 0.22, 0.30, 0.20])
    
    peak_t = rng.uniform(3.0, 5.8, size=(N_sim, 1))
    width_t = rng.uniform(2.5, 4.2, size=(N_sim, 1))
    base_shape = np.exp(-0.5 * ((t_cosmic - peak_t) / width_t)**2)
    
    norm_sfr = (sfr_ms_linear / base_shape[:, -1:]) * (10**rng.normal(0, 0.22, size=(N_sim, 1)))
    sfr_sec = norm_sfr * base_shape
    
    slow_fluct = uniform_filter1d(rng.normal(0, 0.16, size=(N_sim, len(t_cosmic))), size=int(1.0 / dt), axis=1)
    fast_fluct = uniform_filter1d(rng.normal(0, 0.20, size=(N_sim, len(t_cosmic))), size=int(t1 / dt), axis=1)
    sfr_true = np.clip(sfr_sec * (1.0 + slow_fluct + fast_fluct), 1e-4, None)
    
    t_event_lb = np.full(N_sim, 10.0)
    event_strength = np.zeros(N_sim)
    dry_merger_decouple = np.zeros(N_sim, dtype=bool)
    t_dry_merger_lb = np.full(N_sim, 10.0)
    gv_fast_quench = np.zeros(N_sim, dtype=bool)
    
    for i in range(N_sim):
        if pops[i] == 'SB':
            burst_dex = rng.uniform(0.40, 1.8)
            burst_mult = (10**burst_dex - 1.0) * np.exp(-t_lb_hr / t1)
            sfr_true[i] = sfr_true[i] * (1.0 + burst_mult)
            t_event_lb[i] = t1
            event_strength[i] = burst_dex
            
        elif pops[i] == 'GV':
            t_q_start = rng.uniform(0.5, 2.6)
            q_mask = t_lb_hr < t_q_start
            drop_dex = rng.uniform(0.40, 1.30)
            if rng.random() < 0.60:
                gv_fast_quench[i] = True
                tau_q = t1 * rng.uniform(0.8, 1.4)
            else:
                tau_q = t2 * rng.uniform(0.7, 1.2)
            sfr_true[i, q_mask] *= 10**(-drop_dex * (1.0 - np.exp(-(t_q_start - t_lb_hr[q_mask]) / tau_q)))
            t_event_lb[i] = t_q_start
            event_strength[i] = drop_dex
            
        elif pops[i] == 'Q':
            # Quenching barrier + dry mergers tuned to match the GSWLC quiescent plateau
            t_q_start = rng.uniform(3.5, 7.2)
            q_mask = t_lb_hr < t_q_start
            drop_dex = rng.uniform(1.5, 3.2)
            sfr_true[i, q_mask] *= 10**(-drop_dex * (1.0 - np.exp(-(t_q_start - t_lb_hr[q_mask]) / 0.8)))
            t_event_lb[i] = t_q_start
            event_strength[i] = drop_dex
            if rng.random() < 0.48:
                dry_merger_decouple[i] = True
                t_dry_merger_lb[i] = rng.uniform(0.3, 3.2)

    sfr_bins = np.clip(sfr_true[:, bin_idx], 1e-4, None)
    log_sfr_bins = np.log10(sfr_bins)
    ssfr_today = log_sfr_bins[:, 0] - fixed_log_mass
    log_sfr_sec_bins = np.log10(np.clip(sfr_sec[:, bin_idx], 1e-4, None))
    
    pred_med = np.zeros_like(log_sfr_bins)
    sigma_ci = 0.31
    
    if regime == 'lag':
        for i in range(N_sim):
            drift = np.cumsum(rng.normal(0, 0.050, size=len(t_lb_bins)))
            if pops[i] == 'MS':
                pred_med[i] = log_sfr_sec_bins[i] + drift
            elif pops[i] == 'SB':
                lag_capture = np.exp(-t2 / max(t1, 1e-3))
                pred_med[i] = (1.0 - lag_capture) * log_sfr_sec_bins[i] + lag_capture * log_sfr_bins[i] + drift
            elif pops[i] == 'GV':
                if gv_fast_quench[i]:
                    lag_capture = np.exp(-t2 / max(t1, 1e-3))
                    pred_med[i] = (1.0 - lag_capture) * log_sfr_sec_bins[i] + lag_capture * log_sfr_bins[i] + drift
                else:
                    pred_med[i] = log_sfr_bins[i] + drift
                    pre_trans = t_lb_bins >= t_event_lb[i]
                    pred_med[i, pre_trans] += rng.normal(0.55, 0.20, size=np.sum(pre_trans))
            elif pops[i] == 'Q':
                pred_med[i] = log_sfr_bins[i] + drift * 0.75
                pre_q = t_lb_bins >= t_event_lb[i]
                pred_med[i, pre_q] += rng.normal(0.65, 0.25, size=np.sum(pre_q))
                if dry_merger_decouple[i]:
                    dm_mask = t_lb_bins >= t_dry_merger_lb[i]
                    pred_med[i, dm_mask] += rng.normal(0.60, 0.20, size=np.sum(dm_mask))
                    
    elif regime == 'instant':
        t_memory_horizon = t2
        for i in range(N_sim):
            if pops[i] == 'MS':
                # No violent event to destroy information, just secular drift
                drift = np.cumsum(rng.normal(0, 0.050, size=len(t_lb_bins)))
                pred_med[i] = log_sfr_sec_bins[i] + drift
            else:
                # Information is destroyed instantly when leaving the MS (t1 ~= t2)
                steps_past_horizon = np.maximum(0.0, t_lb_bins - t_memory_horizon)
                destruction_rate = 0.45 * (1.0 + 0.8 * event_strength[i])
                scrambling = rng.normal(0, 0.12, size=len(t_lb_bins)) + destruction_rate * np.sqrt(steps_past_horizon) * rng.choice([-1, 1])
                pred_med[i] = log_sfr_bins[i] + scrambling

    is_acc = np.abs(log_sfr_bins - pred_med) <= sigma_ci
    cum_acc = np.minimum.accumulate(is_acc, axis=1)
    cont_mem = np.zeros(N_sim)
    for k in range(len(t_lb_bins)):
        cont_mem[cum_acc[:, k]] = t_lb_bins[k]
        
    return ssfr_today, cont_mem

ssfr_lag, mem_lag   = simulate_sam_fixed_mass(regime='lag',     fixed_log_mass=fixed_log_mass, t1=0.2, t2=2.0)
# ssfr_lag, mem_lag   = simulate_sam_fixed_mass(regime='lag',     fixed_log_mass=fixed_log_mass, t1=2.0, t2=0.2)
ssfr_inst, mem_inst = simulate_sam_fixed_mass(regime='instant', fixed_log_mass=fixed_log_mass, t1=0.2, t2=0.2)

# ==============================================================================
# 3. BINNING & PLOTTING THE 3 CURVES (sSFR vs. Continuous Memory)
# ==============================================================================
ssfr_edges = np.linspace(-13.0, -8.8, 26)
ssfr_centers = 0.5 * (ssfr_edges[:-1] + ssfr_edges[1:])

def compute_binned_stats(x, y, edges, min_count=10):
    mean_arr = np.full(len(edges) - 1, np.nan)
    sem_arr  = np.full(len(edges) - 1, np.nan)
    p25_arr  = np.full(len(edges) - 1, np.nan)
    p75_arr  = np.full(len(edges) - 1, np.nan)
    for k in range(len(edges) - 1):
        mask = (x >= edges[k]) & (x < edges[k + 1])
        if np.sum(mask) >= min_count:
            vals = y[mask]
            mean_arr[k] = np.nanmean(vals)
            sem_arr[k]  = np.nanstd(vals) / np.sqrt(len(vals))
            p25_arr[k]  = np.nanpercentile(vals, 25)
            p75_arr[k]  = np.nanpercentile(vals, 75)
    return mean_arr, sem_arr, p25_arr, p75_arr

real_mean, real_sem, real_p25, real_p75 = compute_binned_stats(ssfr_real_slice, mem_real_slice, ssfr_edges)
lag_mean,  lag_sem,  lag_p25,  lag_p75  = compute_binned_stats(ssfr_lag,        mem_lag,        ssfr_edges)
inst_mean, inst_sem, inst_p25, inst_p75 = compute_binned_stats(ssfr_inst,       mem_inst,       ssfr_edges)

fig, ax = plt.subplots(figsize=(14, 8), dpi=130)

# Shaded background regions for the 4 galaxy populations (matching Section 4.3 cutoffs)
ax.axvspan(-13.0, -11.5, color='#7f7f7f', alpha=0.08, label='Quiescent (< -11.5)')
ax.axvspan(-11.5, -10.5, color='#2ca02c', alpha=0.08, label='Green Valley (-11.5 to -10.5)')
ax.axvspan(-10.5, -9.5,  color='#1f77b4', alpha=0.08, label='Main Sequence (-10.5 to -9.5)')
ax.axvspan(-9.5,  -8.8,  color='#d62728', alpha=0.08, label='Starburst (> -9.5)')

for x_sep in [-11.5, -10.5, -9.5]:
    ax.axvline(x_sep, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)

# 1. Real Data Curve
valid_r = ~np.isnan(real_mean)
ax.plot(ssfr_centers[valid_r], real_mean[valid_r], color='black', lw=3.0, marker='o', markersize=6,
        label=rf'Image Model Prediction ($\log M_* \approx {fixed_log_mass} \pm {mass_window}$)')
ax.fill_between(ssfr_centers[valid_r],
                real_mean[valid_r] - 2 * real_sem[valid_r],
                real_mean[valid_r] + 2 * real_sem[valid_r],
                color='black', alpha=0.18)

# 2. Toy Model: Morphological Lag (t2 >> t1)
valid_l = ~np.isnan(lag_mean)
ax.plot(ssfr_centers[valid_l], gaussian_filter1d(lag_mean[valid_l], sigma=0.8),
        color='#d62728', lw=2.8, ls='--', marker='s', markersize=5,
        label=r'Toy Model: Morphological Lag ($t_2 \gg t_1$)')
        # label=r'Toy Model: SFR Lag ($t_1 \gg t_2$)')
ax.fill_between(ssfr_centers[valid_l],
                gaussian_filter1d(lag_mean[valid_l], sigma=0.8) - 2 * gaussian_filter1d(lag_sem[valid_l], sigma=0.8),
                gaussian_filter1d(lag_mean[valid_l], sigma=0.8) + 2 * gaussian_filter1d(lag_sem[valid_l], sigma=0.8),
                color='#d62728', alpha=0.12)

# 3. Toy Model: Instantaneous Destruction (t1 ~= t2)
valid_i = ~np.isnan(inst_mean)
ax.plot(ssfr_centers[valid_i], gaussian_filter1d(inst_mean[valid_i], sigma=0.8),
        color='#1f77b4', lw=2.8, ls='-.', marker='^', markersize=5,
        label=r'Toy Model: Instant Info Destruction ($t_1 \simeq t_2$)')
ax.fill_between(ssfr_centers[valid_i],
                gaussian_filter1d(inst_mean[valid_i], sigma=0.8) - 2 * gaussian_filter1d(inst_sem[valid_i], sigma=0.8),
                gaussian_filter1d(inst_mean[valid_i], sigma=0.8) + 2 * gaussian_filter1d(inst_sem[valid_i], sigma=0.8),
                color='#1f77b4', alpha=0.15)

ax.set_xlabel(r'$\log\,\mathrm{sSFR}\;[\mathrm{yr}^{-1}]$', fontsize=16)
ax.set_ylabel('Continuous Morphological Memory [Gyr]', fontsize=16)
ax.set_title(rf'Continuous Memory vs. sSFR at Fixed Mass ($\log M_* = {fixed_log_mass}\;M_\odot$)', fontsize=16)
ax.set_xlim(-8.8, -13.0)
ax.set_ylim(-0.2, 8.8)
ax.tick_params(labelsize=14)

# Place the legend completely outside the plot area
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=11.5, frameon=True)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Appendix B — Figure 10: Random Test Set Galaxies Inferred Posteriors

In [ ]:
ranges = [(9, 12), (-14, -7), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't$_{50}$', 'Av', '12+log(O/H)']

conds = {
    'Old Elliptical': (theta[:, 0] > 11.0) & (theta[:, 1] < -2.5) & (theta[:, 4] < 0.5),
    'Young Starburst': (theta[:, 1] > 0.0) & (theta[:, 3] < 3.0),
    'Blue Spiral': (theta[:, 0] > 10.0) & (theta[:, 0] < 10.8) & (theta[:, 1] > -1.0) & (theta[:, 4] < 1.0),
    'Red Spiral': (theta[:, 0] > 10.5) & (theta[:, 1] > -2.5) & (theta[:, 1] < -1.5) & (theta[:, 4] > 1.0),
    'Dusty Edge-on': (theta[:, 4] > 2.0),
    'Green Valley': (theta[:, 1] > -2.0) & (theta[:, 1] < -1.0) & (theta[:, 0] > 10.0)
}

colors = ['firebrick', 'blue', 'cyan', 'salmon', 'saddlebrown', 'olive']
names = list(conds.keys())
test_mask = np.arange(train_index, len(gswlc_data))
gids = []

np.random.seed(177)
for name in names:
    valid_idx = np.where(conds[name])[0]
    valid_test_idx = np.intersect1d(valid_idx, test_mask)
    if len(valid_test_idx) > 0:
        gids.append(np.random.choice(valid_test_idx))
    else:
        print(f"Warning: No match found for {name}. Falling back to a random test galaxy.")
        gids.append(np.random.choice(test_mask))

fig_main = plt.figure(figsize=(24, 16))
subfigs = fig_main.subfigures(2, 3, wspace=0.05, hspace=0.05).flatten()

for i, (gid, name, color) in enumerate(zip(gids, names, colors)):
    xobs_im = gswlc_finetuned_embeds[gid, :]
    x_truth = theta[gid, :].copy()
    x_truth[1] = x_truth[1] - 9  # Adjust sSFR for plotting
    
    # Sample posterior and slice [:, 0:6] for the 6 physical parameters
    posterior_samples_im = posterior_ensemble_finetuned_vit.sample(
        (1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False
    ).cpu().numpy()[:, 0:6]

    posterior_samples_im[:, 1] = posterior_samples_im[:, 1] - 9
    
    corner.corner(
        posterior_samples_im, 
        fig=subfigs[i], 
        truths=x_truth, 
        truth_color='k', 
        color=color, 
        range=ranges, 
        labels=labels, 
        plot_datapoints=False, 
        fill_contours=True,
        quantiles=(0.16, 0.5, 0.84),
        levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)]
    )
    
    for ax in subfigs[i].get_axes():
        if ax.xaxis.label:
            ax.xaxis.label.set_size(14)
        if ax.yaxis.label:
            ax.yaxis.label.set_size(14)
        for tick in ax.get_xticklabels():
            tick.set_fontsize(10)
            tick.set_rotation(45)
        for tick in ax.get_yticklabels():
            tick.set_fontsize(10)

    ax_img = subfigs[i].add_axes([0.65, 0.65, 0.3, 0.3])
    img = Image.open(io.BytesIO(gswlc_images_skyserver[gid]))
    ax_img.imshow(img)
    ax_img.axis('off')

plt.show()

# Appendix C — Figure 11: 11-Band Photometry vs. Image vs. Combined Prediction Precision

In [ ]:
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

n_plot_11b = len(primary_individual_scatters_11_bands["11 Band Photometry"])
x_m_11b = mvals_full if n_plot_11b == len(mvals_full) else mvals
y_s_11b = sfrvals_full if n_plot_11b == len(sfrvals_full) else sfrvals
vit_scatters_11b = primary_individual_scatters["Finetuned ViT"] if len(primary_individual_scatters["Finetuned ViT"]) == n_plot_11b else primary_individual_scatters["Finetuned ViT"][train_index:]

plt.subplots(figsize=(15, 21), nrows=6, ncols=3)
plt.suptitle('11 Band Photometry vs Image vs Combined Prediction Accuracy', fontsize=18, y=1)
for qty in range(6):
    c_lower_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 2)
    c_upper_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 98)
    
    plt.subplot(6, 3, qty*3+1)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry}$', fontsize=18)
    
    plt.subplot(6, 3, qty*3+2)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=vit_scatters_11b[:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)  
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Image}$', fontsize=18)
    plt.tight_layout()

    plt.subplot(6, 3, qty*3+3)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Combined"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry+Image}$', fontsize=18)


plt.show()

In [ ]:
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

n_plot_11b = len(primary_individual_scatters_11_bands["11 Band Combined"])
x_m_11b = mvals_full if n_plot_11b == len(mvals_full) else mvals
y_s_11b = sfrvals_full if n_plot_11b == len(sfrvals_full) else sfrvals
vit_scatters_11b = primary_individual_scatters["Finetuned ViT"] if len(primary_individual_scatters["Finetuned ViT"]) == n_plot_11b else primary_individual_scatters["Finetuned ViT"][train_index:]

plt.subplots(figsize=(15, 21), nrows=6, ncols=3)
plt.suptitle('11 Band Photometry vs Image vs Combined Prediction Accuracy', fontsize=18, y=1)
for qty in range(6):
    c_lower_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 2)
    c_upper_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 98)
    
    plt.subplot(6, 3, qty*3+1)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry}$', fontsize=18)
    
    plt.subplot(6, 3, qty*3+2)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=vit_scatters_11b[:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)  
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Image}$', fontsize=18)
    plt.tight_layout()

    plt.subplot(6, 3, qty*3+3)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Combined"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry+Image}$', fontsize=18)

plt.show()

# Appendix D — Figure 12: 11-Band Photometry, Image, and Combined Inferred Posteriors

In [ ]:
ranges = [(9, 12), (-14, -7), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

np.random.seed(256)
random_gal = np.random.randint(train_index, len(gswlc_data), size=(1))
gid = int(random_gal[0])

xobs_sed = eleven_bands_SED[gid, :]
xobs_im = gswlc_finetuned_embeds[gid, :]
x_truth_raw_11_band = theta[gid, :].copy()
x_truth_11_band = x_truth_raw_11_band

x_truth_11_band[1] = x_truth_raw_11_band[1] - 9 
print(x_truth_11_band[1])


posterior_samples_sed = posterior_ensemble_sed_11_bands.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_im = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_combined = posterior_ensemble_combined_11_bands.sample((1000,), torch.Tensor(np.hstack((xobs_im, xobs_sed))).to('cuda:1'), show_progress_bars=False).cpu().numpy()[:, 0:6]

plt.imshow(Image.open(io.BytesIO(gswlc_images_skyserver[gid])))
plt.axis('off')
plt.figure()
plt.scatter(np.arange(11), xobs_sed)
plt.xticks(np.arange(0, 11), ['FUV', 'NUV', 'u', 'g', 'r', 'i', 'z', 'W1', 'W2', 'W3', 'W4'], fontsize=16)
plt.xlabel('Band', fontsize=18)
plt.ylabel('Magnitude', fontsize=18)

posterior_samples_sed[:,1] = posterior_samples_sed[:,1] - 9
posterior_samples_im[:,1] = posterior_samples_im[:,1] - 9
posterior_samples_combined[:,1] = posterior_samples_combined[:,1] - 9 

fig = corner.corner(posterior_samples_sed, truths=x_truth, color='dodgerblue', range=ranges, labels=labels, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k', color='firebrick', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_combined, truths=x_truth, truth_color='k', color='green', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])

sed_legend_patch = mpatches.Patch(color='dodgerblue', label='Photometry')
image_legend_patch = mpatches.Patch(color='firebrick', label='Image')
combined_legend_patch = mpatches.Patch(color='green', label='Combined')
fig.legend(handles=[sed_legend_patch, image_legend_patch, combined_legend_patch],
           loc='upper right', fontsize='18', title='Posterior Source', frameon=True, title_fontsize='18')
fig.set_size_inches(12, 12)
for label in fig.get_axes():
    label.xaxis.label.set_size(18)
    label.yaxis.label.set_size(18)
    for tick in label.get_xticklabels():
        tick.set_fontsize(14)
    for tick in label.get_yticklabels():
        tick.set_fontsize(14)
plt.show()

# Appendix E — Figure 13: Population-Level Full Degeneracies (Centered Residuals)

In [ ]:
# import matplotlib.lines as mlines
# import numpy as np
# import matplotlib.pyplot as plt
# import corner

# # 1. Calculate median standard deviation for each parameter (sigma = (p84 - p16) / 2)
# # This uses your pre-calculated lightweight arrays instead of raw samples
# std_bands = np.nanmedian((sed5_p84[:, 0:6] - sed5_p16[:, 0:6]) / 2.0, axis=0)
# std_combined = np.nanmedian((comb5_p84[:, 0:6] - comb5_p16[:, 0:6]) / 2.0, axis=0)
# std_images = np.nanmedian((im_p84[:, 0:6] - im_p16[:, 0:6]) / 2.0, axis=0)

# # 2. Extract the offset 5x5 Spearman correlation matrices (rho)
# rho_bands = np.nanmedian(covs_5_band, axis=0)
# rho_combined = np.nanmedian(covs_combined_5_band, axis=0)
# rho_images = np.nanmedian(covs_image, axis=0)

# # 3. Build true 6x6 symmetric covariance matrices: Cov(X, Y) = rho * std(X) * std(Y)
# def build_true_cov_matrix(std_arr, rho_matrix):
#     cov = np.zeros((6, 6))
#     for i in range(6):
#         cov[i, i] = std_arr[i] ** 2  # Variance on the diagonal
#         for j in range(i):
#             r = rho_matrix[i-1, j]   # Shift index back to match your 5x5 matrix
#             cov[i, j] = r * std_arr[i] * std_arr[j]
#             cov[j, i] = cov[i, j]
#     return cov

# true_cov_bands = build_true_cov_matrix(std_bands, rho_bands)
# true_cov_combined = build_true_cov_matrix(std_combined, rho_combined)
# true_cov_images = build_true_cov_matrix(std_images, rho_images)

# # 4. Generate synthetic pooled residuals directly from the true covariance matrices
# n_params = 6
# n_mock_samples = 150000
# res_bands = np.random.multivariate_normal(mean=np.zeros(n_params), cov=true_cov_bands, size=n_mock_samples)
# res_combined = np.random.multivariate_normal(mean=np.zeros(n_params), cov=true_cov_combined, size=n_mock_samples)
# res_images = np.random.multivariate_normal(mean=np.zeros(n_params), cov=true_cov_images, size=n_mock_samples)

# # 5. Setup Plotting Styling
# labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']
# color_bands = 'dodgerblue'
# color_combined = 'green'
# color_images = 'red'

# kwargs_bands = {
#     "labels": labels, "label_kwargs": {"fontsize": 32},
#     "hist_kwargs": {"density": True, "linewidth": 2, "histtype": "stepfilled", "alpha": 0.3},
#     "plot_datapoints": False, "plot_density": True, "fill_contours": True,
#     "alpha": 0.3, "levels": (0.68, 0.95), "bins": 30,                   
# }
# kwargs_combined = {
#     "labels": labels, "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "-"},
#     "plot_datapoints": False, "plot_density": False, "fill_contours": False,       
#     "contour_kwargs": {"linewidths": 2.5, "linestyles": "-"}, "levels": (0.68, 0.95), "bins": 30,
# }
# kwargs_images = {
#     "labels": labels, "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "--"},
#     "plot_datapoints": False, "plot_density": False, "fill_contours": False,       
#     "contour_kwargs": {"linewidths": 2.5, "linestyles": "--"}, "levels": (0.68, 0.95), "bins": 30,
# }

# # 6. Generate the corner plot
# fig = plt.figure(figsize=(20, 20))
# fig = corner.corner(res_bands, fig=fig, color=color_bands, **kwargs_bands)
# corner.corner(res_combined, fig=fig, color=color_combined, **kwargs_combined)
# corner.corner(res_images, fig=fig, color=color_images, **kwargs_images)

# fig.subplots_adjust(wspace=0.2, hspace=0.2)
# axes = np.array(fig.axes).reshape((n_params, n_params))
# bbox_props = dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none", alpha=0.8)

# for yi in range(n_params):
#     for xi in range(yi):
#         ax = axes[yi, xi]
#         ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
#         ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
#         ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
#         # Read the rho value directly from the offset 5x5 Spearman matrices
#         rho_val_bands = rho_bands[yi-1, xi]
#         rho_val_comb = rho_combined[yi-1, xi]
#         rho_val_images = rho_images[yi-1, xi]
        
#         txt_bands = f"$\\rho_{{phot}} = {rho_val_bands:.2f}$"
#         txt_images = f"$\\rho_{{images}} = {rho_val_images:.2f}$"
#         txt_comb = f"$\\rho_{{comb}} = {rho_val_comb:.2f}$"

#         ax.text(0.05, 0.85, txt_bands, transform=ax.transAxes, 
#                 fontsize=18, color=color_bands, ha='left', va='bottom', weight='bold', bbox=bbox_props)
#         ax.text(0.05, 0.75, txt_images, transform=ax.transAxes, 
#                 fontsize=18, color=color_images, ha='left', va='bottom', weight='bold', bbox=bbox_props)
#         ax.text(0.05, 0.65, txt_comb, transform=ax.transAxes, 
#                 fontsize=18, color=color_combined, ha='left', va='bottom', weight='bold', bbox=bbox_props)
    
#     axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

# legend_handles = [
#     mlines.Line2D([], [], color=color_bands, lw=4, alpha=0.6, label='5 Band Photometry'),
#     mlines.Line2D([], [], color=color_images, lw=3, linestyle='--', label='Image'),
#     mlines.Line2D([], [], color=color_combined, lw=3, linestyle='-', label='Combined')
# ]

# fig.axes[n_params - 1].legend(handles=legend_handles, loc='center left', fontsize=34, frameon=False)
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/population_full_degeneracies.pdf', dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
FAST_MODE = True
STRIDE = 100       

# 1. Compute pooled residuals on the 6 physical parameters ([:, :, 0:6])
res_bands = get_pooled_residuals(all_samples_5_band[:, :, 0:6])
res_combined = get_pooled_residuals(all_samples_combined_5_band[:, :, 0:6])
res_images = get_pooled_residuals(all_samples_image[:, :, 0:6])

if FAST_MODE:
    print(f"Running in FAST_MODE. Plotting 1/{STRIDE} of the samples...")
    res_bands = res_bands[::STRIDE]
    res_combined = res_combined[::STRIDE]
    res_images = res_images[::STRIDE]

# 2. Extract Pre-calculated Median Covariances ([:, 0:5, 0:5])
median_cov_bands = np.nanmedian(all_covs_5_band[:, 0:5, 0:5], axis=0)
median_cov_combined = np.nanmedian(all_covs_combined_5_band[:, 0:5, 0:5], axis=0)
median_cov_images = np.nanmedian(all_covs_image[:, 0:5, 0:5], axis=0)

# 3. Setup plotting parameters
n_params = res_bands.shape[-1]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

color_bands = 'dodgerblue'
color_combined = 'green'
color_images = 'red'

kwargs_bands = {
    "labels": labels,
    "label_kwargs": {"fontsize": 32},
    "hist_kwargs": {"density": True, "linewidth": 2, "histtype": "stepfilled", "alpha": 0.3},
    "plot_datapoints": False,
    "plot_density": True,
    "fill_contours": True,
    "alpha": 0.3,                 
    "levels": (0.68, 0.95),         
    "bins": 30,                   
}

kwargs_combined = {
    "labels": labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "-"},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5, "linestyles": "-"}, 
    "levels": (0.68, 0.95),           
    "bins": 30,
}

kwargs_images = {
    "labels": labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "--"},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5, "linestyles": "--"}, 
    "levels": (0.68, 0.95),           
    "bins": 30,
}

# 4. Generate the corner plot
fig = plt.figure(figsize=(20, 20))
fig = corner.corner(res_bands, fig=fig, color=color_bands, **kwargs_bands)
corner.corner(res_combined, fig=fig, color=color_combined, **kwargs_combined)
corner.corner(res_images, fig=fig, color=color_images, **kwargs_images)

fig.subplots_adjust(wspace=0.2, hspace=0.2)

axes = np.array(fig.axes).reshape((n_params, n_params))
bbox_props = dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none", alpha=0.8)

for yi in range(n_params):
    for xi in range(yi):
        ax = axes[yi, xi]
        ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
        rho_bands = median_cov_bands[yi - 1, xi]
        rho_comb = median_cov_combined[yi - 1, xi]
        rho_images = median_cov_images[yi - 1, xi]
        
        txt_bands = f"$\\rho_{{phot}} = {rho_bands:.2f}$"
        txt_images = f"$\\rho_{{images}} = {rho_images:.2f}$"
        txt_comb = f"$\\rho_{{comb}} = {rho_comb:.2f}$"

        ax.text(0.05, 0.85, txt_bands, transform=ax.transAxes, 
                fontsize=18, color=color_bands, ha='left', va='bottom', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.75, txt_images, transform=ax.transAxes, 
                fontsize=18, color=color_images, ha='left', va='bottom', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.65, txt_comb, transform=ax.transAxes, 
                fontsize=18, color=color_combined, ha='left', va='bottom', weight='bold', bbox=bbox_props)
    
    axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

legend_handles = [
    mlines.Line2D([], [], color=color_bands, lw=4, alpha=0.6, label='5 Band Photometry'),
    mlines.Line2D([], [], color=color_images, lw=3, linestyle='--', label='Image'),
    mlines.Line2D([], [], color=color_combined, lw=3, linestyle='-', label='Combined')
]
fig.axes[n_params - 1].legend(handles=legend_handles, loc='center left', fontsize=34, frameon=False)
plt.show()

# Appendix F — Figure 14: Individual Galaxies SFHs with Confidence Intervals

In [ ]:
gal_mass = mvals_full[slice_im]
gal_sfr  = sfrvals_full[slice_im]
gal_ssfr = gal_sfr - gal_mass
images_sub = gswlc_images_skyserver[slice_im]

def safe_log(arr):
    return np.log10(np.clip(arr, 1e-4, None))

np.random.seed(144)
# np.random.seed(256)

fig, axes = plt.subplots(4, 3, figsize=(18, 16), sharex=True, sharey='row')
common_lookback_grid = np.linspace(0, 10.0, 200)

print("Generating 10 Gyr Reconstruction Grid (Single Galaxy CIs + Images)...")

for row_idx, ssfr_idx in enumerate([3, 2, 1, 0]): 
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        
        m_min, m_max = mass_bins[col_idx], mass_bins[col_idx+1]
        s_min, s_max = ssfr_bins[ssfr_idx], ssfr_bins[ssfr_idx+1]
        
        bin_mask = (gal_mass >= m_min) & (gal_mass < m_max) & \
                   (gal_ssfr >= s_min) & (gal_ssfr < s_max)
        
        indices = np.where(bin_mask)[0]
        n_count = len(indices)
        bin_title = f"{mass_labels[col_idx]}\n{ssfr_labels[ssfr_idx]}\n(Total Bin N={n_count})"

        if n_count == 0:
            ax.text(0.5, 0.5, "No Galaxies", ha='center', transform=ax.transAxes)
            ax.set_title(bin_title, fontsize=10)
            continue

        rand_gal_idx = np.random.choice(indices)

        # Process True SFH
        sfr_true = sfh_mod_sub[rand_gal_idx, 0, :]  
        t_cosmic = sfh_mod_sub[rand_gal_idx, 1, :]
        t_obs = np.max(t_cosmic) 
        t_lookback_gal = t_obs - t_cosmic
        
        sort_order = np.argsort(t_lookback_gal)
        true_sfr_interp = np.interp(common_lookback_grid, 
                                    t_lookback_gal[sort_order], 
                                    sfr_true[sort_order])

        # Process Predicted Posterior Samples (directly in log SFR space from 34D theta)
        pred_med, pred_p16, pred_p84 = [], [], []
        for t in reconstruction_times:
            feature_idx = time_to_idx_35d[t]
            # full_sfr_samples = all_samples_image[rand_gal_idx, :, feature_idx]
            full_sfr_samples = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(gswlc_finetuned_embeds[slice_im][rand_gal_idx]).to(device), show_progress_bars=False).cpu().numpy()[:, feature_idx]
            pred_med.append(np.nanpercentile(full_sfr_samples, 50))
            pred_p16.append(np.nanpercentile(full_sfr_samples, 16))
            pred_p84.append(np.nanpercentile(full_sfr_samples, 84))

        pred_med = np.array(pred_med)
        pred_p16 = np.array(pred_p16)
        pred_p84 = np.array(pred_p84)

        ax.plot(common_lookback_grid, safe_log(true_sfr_interp), color='black', lw=2, label='True SFH')
        ax.fill_between(reconstruction_times, pred_p16, pred_p84, 
                        color='red', alpha=0.3, label='Predicted 16-84% CI')
        ax.plot(reconstruction_times, pred_med, color='red', 
                linestyle='--', marker='o', markersize=3, label='Predicted Median')

        # Image Inset
        axins = ax.inset_axes([0.65, 0.65, 0.3, 0.3])
        img_bytes = images_sub[rand_gal_idx]
        img_data = np.array(Image.open(io.BytesIO(img_bytes)))
        axins.imshow(img_data, origin='lower') 
        axins.axis('off')
        for spine in axins.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(0.5)

        ax.set_title(bin_title, fontsize=14)
        ax.set_xlim(-0.2, 10.2)
        
        if row_idx == 0: ax.set_ylim(-4.5, 2)
        if row_idx == 1: ax.set_ylim(-2, 2)
        if row_idx == 2: ax.set_ylim(-2, 2)
        if row_idx == 3: ax.set_ylim(-4, 2)
        
        ax.grid(alpha=0.3)
        ax.set_xlabel("Lookback Time [Gyr]", fontsize=18)
        ax.set_ylabel("log SFR [$M_\odot$/yr]", fontsize=18)
        ax.tick_params(labelbottom=True, labelleft=True)
        
        if row_idx == 0 and col_idx == 0:
            ax.legend(loc='lower left', fontsize=14)

plt.tight_layout()
plt.show()

# Appendix G — Figure 15: Deepest Accurate SFH Prediction (Robust Morphological Memory)

In [ ]:
num_gals = len(x_mass_im)
num_times = len(timescale_map)

print("Calculating Robust Memory Depth from 'individual_accurate_cami' dictionary...")

raw_accuracy_matrix = np.zeros((num_gals, num_times), dtype=bool)
for i, (t_val, key) in enumerate(timescale_map):
    if key in individual_accurate_cami:
        valid_indices = individual_accurate_cami[key]
        raw_accuracy_matrix[valid_indices, i] = True

robust_accuracy_matrix = np.zeros_like(raw_accuracy_matrix)
for i in range(1, num_times - 1):
    robust_accuracy_matrix[:, i] = (
        raw_accuracy_matrix[:, i - 1] & 
        raw_accuracy_matrix[:, i] & 
        raw_accuracy_matrix[:, i + 1]
    )

robust_accuracy_matrix[:, 0] = raw_accuracy_matrix[:, 0] & raw_accuracy_matrix[:, 1]
robust_accuracy_matrix[:, -1] = raw_accuracy_matrix[:, -1] & raw_accuracy_matrix[:, -2]

robust_memory_depth = np.zeros(num_gals)
for i, (t_val, key) in enumerate(timescale_map):
    robust_indices = np.where(robust_accuracy_matrix[:, i])[0]
    robust_memory_depth[robust_indices] = t_val

fig, ax = plt.subplots(figsize=(12, 9))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_mass_im, y_sfr_im, C=robust_memory_depth, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=10.0, edgecolors='none')

cb = fig.colorbar(hb, ax=ax)
cb.set_label('Lookback Time [Gyr]', fontsize=18) 
cb.set_ticks([0, 2, 4, 6, 8, 10])

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Deepest Accurate SFH Prediction', fontsize=18)
ax.legend(loc='lower right', frameon=False, labelcolor='black') 
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
plt.show()

# Appendix H — Figure 16: Continuous SFH Memory from 11-Band and 5-Band Photometry Models

In [ ]:
# Compute SFH Memory Dictionaries for 11-Band and 5-Band Photometry Models
individual_accurate_11_bands, cumulative_accurate_11_bands, failed_at_step_11_bands = compute_sfh_memory_dicts(sed11_p16,sed11_p50,sed11_p84, theta_full_sfh_sfr)

individual_accurate_5_bands, cumulative_accurate_5_bands, failed_at_step_5_bands = compute_sfh_memory_dicts(sed5_p16,sed5_p50,sed5_p84, theta_full_sfh_sfr)

In [ ]:
# Figure 16 (Top Panel): Continuous SFH Memory (11-Band Photometry)
x_m_11b = mvals_full
y_s_11b = sfrvals_full
memory_depth_11_bands = np.zeros(len(x_m_11b))
for t_val, key in timescale_map:
    if key in cumulative_accurate_11_bands:
        memory_depth_11_bands[cumulative_accurate_11_bands[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_m_11b, y_s_11b, C=memory_depth_11_bands, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=6, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction (11 Band Photometry)', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
plt.show()

In [ ]:
# Figure 16 (Bottom Panel): Continuous SFH Memory (5-Band Photometry)
memory_depth_5_bands = np.zeros(len(x_m_5b))
for t_val, key in timescale_map:
    if key in cumulative_accurate_5_bands:
        memory_depth_5_bands[cumulative_accurate_5_bands[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

# y_ax = y_s_5b
y_ax = tempssfr + theta[:,0] - 9

hb = ax.hexbin(x_m_5b, y_ax, C=memory_depth_5_bands, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=6, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction (5 Band Photometry)', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
plt.show()